# B2 direct-critic readiness probes (§1893)
Executes the frozen probe plan `docs/plans/2026-10-05-b2-critic-probes-plan.md` (peer-reviewed): P0 adjacent-rung floor, P1 convergence ledger + 2× continuation, P2 0.5×/2× width probes. Four foreground runs, sequential. Do not edit thresholds — they are frozen in the plan.

Preconditions: L4 GPU runtime, Drive mounted, this repo checked out on branch `opencode/b2-critic-probes`.

In [ ]:
import subprocess, sys
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:]); print(r.stderr[-1000:], file=sys.stderr)
    return r
r = sh('nvidia-smi --query-gpu=name --format=csv,noheader')
assert 'L4' in r.stdout, 'L4 required, got: ' + r.stdout
import torch; assert torch.cuda.is_available(), 'torch.cuda unavailable'
print('torch', torch.__version__, '| GPU ok')

In [ ]:
import hashlib
DRIVE = '/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production'
RUN_ID = 'droid100_adjoint_v2_5seeds_20261001T080821Z'
OUT = '/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production/runs/b2_critic_probes_20261005'
import os
cache = os.path.join(DRIVE, 'runs', RUN_ID, 'cache')
assert os.path.isdir(cache), 'B2 cache missing on Drive: ' + cache
cfg = os.path.join(DRIVE, 'runs', RUN_ID, 'config', 'run_config.json')
assert os.path.isfile(cfg), 'run config missing: ' + cfg
ckpt0 = os.path.join(DRIVE, 'runs', RUN_ID, 'jobs', 'seed_0', 'dynamics', 'best.pt')
assert os.path.isfile(ckpt0), 'teacher checkpoint missing: ' + ckpt0
# Code identity: probe flags must exist in the checked-out script.
src = open('scripts/run_allocator_optimization_benchmark.py').read()
for flag in ('--critic-width-scale', '--critic-only', '--probe-panels', 'adjacent_rung_panels', 'surrogate_ledger'):
    assert flag in src, 'missing probe code: ' + flag
head = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('HEAD', head, '| data + code verified')
!mkdir -p "$OUT"/probe_w10 "$OUT"/probe_w05 "$OUT"/probe_w20 "$OUT"/probe_cont

## Run `w10`: width 1.0×, 2500 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 1.0 --eval-checkpoints 300 1000 2500 --max-steps 2500 --output-dir "$OUT"/probe_w10 --local-cache /content/cache 2>&1 | tee "$OUT"/probe_w10.log

## Run `w05`: width 0.5×, 2500 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 0.5 --eval-checkpoints 300 1000 2500 --max-steps 2500 --output-dir "$OUT"/probe_w05 --local-cache /content/cache 2>&1 | tee "$OUT"/probe_w05.log

## Run `w20`: width 2.0×, 2500 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 2.0 --eval-checkpoints 300 1000 2500 --max-steps 2500 --output-dir "$OUT"/probe_w20 --local-cache /content/cache 2>&1 | tee "$OUT"/probe_w20.log

## Run `cont`: width 1.0×, 5000 steps

In [ ]:
!python -u scripts/run_allocator_optimization_benchmark.py --critic-only --probe-panels --critic-width-scale 1.0 --eval-checkpoints 300 1000 2500 5000 --max-steps 5000 --output-dir "$OUT"/probe_cont --local-cache /content/cache 2>&1 | tee "$OUT"/probe_cont.log

## Verification (all four summaries)

In [ ]:
import hashlib, json, os
for tag in ('w10', 'w05', 'w20', 'cont'):
    p = os.path.join(OUT, f'probe_{tag}', 'b2_2_allocator_optimization_summary.json')
    raw = open(p, 'rb').read()
    s = json.loads(raw)
    assert s['seed_finite'] == [True]*5, (tag, s['seed_finite'])
    assert 'surrogate_ledger' in s and 'val_checkpoints' in s, tag
    last = s['eval_checkpoints'][-1]
    v = s['val_checkpoints'][str(last)]['mean_regrets']
    print(f"{tag}: sha256={hashlib.sha256(raw).hexdigest()[:16]} "
          f"width={s['critic_width_scale']} steps={last} "
          f"val critic={v['allocator_critic']:.5f} random={v['random_expected']:.5f} "
          f"unc={v['uncertainty']:.5f} mode0={v['always_mode0']:.5f}")
print('All four summaries present, finite, and structurally complete.')